# Price Data


In [ ]:
import pandas as pd

# Download CSV file from GitHub and read the data
url = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/MANA_price.csv'
data = pd.read_csv(url)

# Standardize the date format
data['Date'] = pd.to_datetime(data['Date'])

# Remove dollar signs and convert to float
for column in ['Open*', 'High', 'Low', 'Close**']:
    data[column] = data[column].replace('[\$,]', '', regex=True).astype(float)

# Calculate typical price and round to 4 decimal places
data['Typical Price'] = ((data['High'] + data['Low'] + data['Close**']) / 3).round(4)

# Create a new DataFrame containing only Date and Typical Price
result = data[['Date', 'Typical Price']]

# Sort by date in ascending order
result = result.sort_values(by='Date')

# Save the result to a new CSV file
result.to_csv('MANA_typical_price.csv', index=False)

# Token Return

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Read CSV file from GitHub
url = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/D/MANA_typical_price.csv'
data = pd.read_csv(url)

# Ensure the Date column is in datetime format
data['Date'] = pd.to_datetime(data['Date'])

# Sort by date (in case dates are not in order)
data = data.sort_values('Date')

# Calculate daily returns
data['Log Price'] = np.log(data['Typical Price'])
data['Return'] = data['Log Price'].diff()

# Drop NaN values (because the first day has no previous day's data)
data = data.dropna()

# Save the results to a new CSV file
output_path = 'token_return.csv'
data.to_csv(output_path, index=False)

# Display the first few rows to confirm
data.head()

,Date,Typical Price,Log Price,Return
1,2023-06-16,0.3351,-1.093326,0.011405
2,2023-06-17,0.3412,-1.075286,0.018040
3,2023-06-18,0.3385,-1.083231,-0.007945
4,2023-06-19,0.3372,-1.087079,-0.003848
5,2023-06-20,0.3438,-1.067695,0.019384


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

# Read CSV file
price_data = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/token_return.csv')

# Ensure Date column is in datetime format
price_data['Date'] = pd.to_datetime(price_data['Date'])

# Sort by date
price_data = price_data.sort_values('Date')

# Visualization
fig = go.Figure()

# Add actual return data
fig.add_trace(go.Scatter(x=price_data['Date'], y=price_data['Return'],
                         mode='lines', name='Return', line=dict(color='blue')))

# Update layout
fig.update_layout(
    title='Daily Token Return',
    xaxis_title='Date(d)',
    yaxis_title='Token Return',
    font=dict(
        family='Times New Roman',
        size=14
    ),
    width=1000,  # Width in pixels
    height=625,
    yaxis=dict(range=[min(price_data['Return']) * 1.1, max(price_data['Return']) * 1.1])  # Adjust y-axis range
)

# Display the chart
fig.show()

/usr/local/lib/python3.10/dist-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



# Prediction

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

# Read CSV files
price_data = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/token_return.csv')
sentiment_data = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/daily_processed_scores.csv')
market_data = pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/D/price.csv')

# Ensure Date columns are in datetime format
price_data['Date'] = pd.to_datetime(price_data['Date'])
sentiment_data['Date'] = pd.to_datetime(sentiment_data['Date'])
market_data['Date'] = pd.to_datetime(market_data['Date'])

# Sort by date
price_data = price_data.sort_values('Date')
sentiment_data = sentiment_data.sort_values('Date')
market_data = market_data.sort_values('Date')

# Merge datasets
data = pd.merge(price_data, sentiment_data, on='Date')
data = pd.merge(data, market_data, on='Date')

# Extract features
features = ['Return', 'Volume', 'Market Cap', 'Sentiment_Score']

# Normalize data
scaler = MinMaxScaler(feature_range=(0, 1))

# Function to create training dataset
def create_dataset(data, time_step=1):
    X, Y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step), :])
        Y.append(data[i + time_step, 0])
    return np.array(X), np.array(Y)

# Use multi-feature model
scaled_data = scaler.fit_transform(data[features].values)
X_multi, Y_multi = create_dataset(scaled_data, 10)

# Define K-Fold cross validation
kf = KFold(n_splits=8, shuffle=True, random_state=1)

# Store evaluation results
mse_prices, mae_prices, r2_prices = [], [], []
mse_multis, mae_multis, r2_multis = [], [], []

# K-Fold cross validation
for train_index, test_index in kf.split(X_multi):
    X_train, X_test = X_multi[train_index], X_multi[test_index]
    Y_train, Y_test = Y_multi[train_index], Y_multi[test_index]

    # Reshape data to fit LSTM input format
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

    # Build and train LSTM model (multiple features)
    model_multi = Sequential()
    model_multi.add(LSTM(50, return_sequences=True, input_shape=(10, X_multi.shape[2])))
    model_multi.add(LSTM(50, return_sequences=False))
    model_multi.add(Dense(25))
    model_multi.add(Dense(1))
    model_multi.compile(optimizer='adam', loss='mean_squared_error')
    model_multi.fit(X_train, Y_train, batch_size=1, epochs=10, verbose=0)

    # Predict
    predictions_multi = model_multi.predict(X_test)
    predictions_multi = scaler.inverse_transform(np.concatenate((predictions_multi, np.zeros((predictions_multi.shape[0], X_multi.shape[2]-1))), axis=1))[:,0]

    # Calculate evaluation metrics
    mse_multi = mean_squared_error(Y_test, predictions_multi)
    mae_multi = mean_absolute_error(Y_test, predictions_multi)
    r2_multi = r2_score(Y_test, predictions_multi)

    mse_multis.append(mse_multi)
    mae_multis.append(mae_multi)
    r2_multis.append(r2_multi)

    # Model using only historical price data
    scaled_price_data = scaler.fit_transform(price_data[['Return']].values)
    X_price, Y_price = create_dataset(scaled_price_data, 10)

    # Split training and testing data
    X_train_price, X_test_price = X_price[train_index], X_price[test_index]
    Y_train_price, Y_test_price = Y_price[train_index], Y_price[test_index]

    # Reshape data to fit LSTM input format
    X_train_price = X_train_price.reshape(X_train_price.shape[0], X_train_price.shape[1], 1)
    X_test_price = X_test_price.reshape(X_test_price.shape[0], X_test_price.shape[1], 1)

    # Build and train LSTM model (using only historical price)
    model_price = Sequential()
    model_price.add(LSTM(50, return_sequences=True, input_shape=(10, 1)))
    model_price.add(LSTM(50, return_sequences=False))
    model_price.add(Dense(25))
    model_price.add(Dense(1))
    model_price.compile(optimizer='adam', loss='mean_squared_error')
    model_price.fit(X_train_price, Y_train_price, batch_size=1, epochs=10, verbose=0)

    # Predict
    predictions_price = model_price.predict(X_test_price)
    predictions_price = scaler.inverse_transform(np.concatenate((predictions_price, np.zeros((predictions_price.shape[0], X_multi.shape[2]-1))), axis=1))[:,0]

    # Calculate evaluation metrics
    mse_price = mean_squared_error(Y_test_price, predictions_price)
    mae_price = mean_absolute_error(Y_test_price, predictions_price)
    r2_price = r2_score(Y_test_price, predictions_price)

    mse_prices.append(mse_price)
    mae_prices.append(mae_price)
    r2_prices.append(r2_price)

# Calculate average evaluation metrics
mean_mse_price = np.mean(mse_prices)
mean_mae_price = np.mean(mae_prices)
mean_r2_price = np.mean(r2_prices)

mean_mse_multi = np.mean(mse_multis)
mean_mae_multi = np.mean(mae_multis)
mean_r2_multi = np.mean(r2_multis)

print(f'MSE (Price Only Prediction): {mse_prices}')
print(f'MAE (Price Only Prediction): {mae_prices}')
print(f'R² (Price Only Prediction): {r2_prices}')

print(f'MSE (Price + Sentiment Prediction): {mse_multis}')
print(f'MAE (Price + Sentiment Prediction): {mae_multis}')
print(f'R² (Price + Sentiment Prediction): {r2_multis}')

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 793ms/step


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 492ms/step


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 319ms/step


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 291ms/step


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 305ms/step
MSE (仅价格预测): [0.5421849310849597, 0.5739560829587504, 0.531404839328992, 0.5005773118132905, 0.48889097084885375, 0.5217782897555616, 0.46339792463109736, 0.5248135923185203]
MAE (仅价格预测): [0.7316252692788449, 0.7521036521022665, 0.7158708858515274, 0.7010094193001662, 0.6932964596367641, 0.7162888734746632, 0.6753645254066335, 0.7181215493771901]
R² (仅价格预测): [-76.67016696835951, -67.66732722268961, -26.70085942909757, -53.52515428890478, -58.81066859363437, -57.974660818537956, -61.49458984154127, -56.26151719272426]
MSE (价格+情感预测): [0.5673649523595485, 0.5058244660341403, 0.4928635995364159, 0.49948790612858834, 0.5563745572784508, 0.5444368004308544, 0.48895673584851385, 0.47412511375771343]
MAE (价格+情感预测): [0.7489317786242462, 0.7043827160946283, 0.695065638689324, 0.6948342804400972, 0.7417691433942216, 0.7321738966988882, 0.6911946725191019, 0.684247610536742]
R² (价格+情感预测): [-89.14477723295408, -49.953507127879206, -47.50219634519658, -27.90772

In [ ]:
import plotly.graph_objects as go

# Create the plot
fig = go.Figure()

# Add actual and predicted data
fig.add_trace(go.Scatter(x=valid['Date'], y=valid['Return'],
                         mode='lines', name='Actual', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=valid['Date'], y=valid['Predictions_Price'],
                         mode='lines', name='Predictions (Price Only)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=valid['Date'], y=valid['Predictions_Multi'],
                         mode='lines', name='Predictions (Multiple Features)', line=dict(color='green')))

# Update layout
fig.update_layout(
    title='Prediction Models VS Actual',
    xaxis_title='Date(d)',
    yaxis_title='Token Return',
    font=dict(
        family='Times New Roman',
        size=12
    ),
    width=1000,  # Width in pixels
    height=625,
    yaxis=dict(range=[-0.05, 0.05])  # Adjust y-axis range
)

# Display the chart
fig.show()